# clustering_squidpy parameter tuning

This notebook calls the same functions used by the Nextflow `CLUSTERING_SQUIDPY` stage so parameters can be adjusted interactively before committing them to the pipeline config.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

from IPython.display import Image, display

from merxen.analysis.clustering_squidpy import (
    load_spatialdata_adata,
    plot_qc_histograms,
    plot_spatial_scatter,
    plot_umap,
    run_clustering_squidpy,
    run_scanpy_clustering,
    save_clustered_adata,
    save_qc_metrics,
)
from merxen.config import (
    ClusteringSquidpyConfig,
    ClusteringSquidpySampleConfig,
)

In [ ]:
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

pair_id = "P7513"
results_dir = PROJECT_ROOT / "results"
output_dir = results_dir / pair_id / "clustering_squidpy_interactive"

samples = [
    ClusteringSquidpySampleConfig(
        sample_id=f"{pair_id}_MERSCOPE",
        platform="MERSCOPE",
        zarr_path=results_dir
        / pair_id
        / "merscope"
        / "latest"
        / "latest_spatialdata.zarr",
    ),
    ClusteringSquidpySampleConfig(
        sample_id=f"{pair_id}_XENIUM",
        platform="XENIUM",
        zarr_path=results_dir
        / pair_id
        / "xenium"
        / "latest"
        / "latest_spatialdata.zarr",
    ),
]

cfg = ClusteringSquidpyConfig(
    pair_id=pair_id,
    output_dir=output_dir,
    samples=samples,
    min_counts=10,
    min_cells=5,
    normalize_target_sum=None,
    n_pcs=50,
    n_neighbors=15,
    leiden_resolution=1.0,
    random_seed=0,
    spatial_point_size=2.0,
    figure_dpi=160,
)
cfg

Load one sample and build the Squidpy-ready AnnData object.

In [ ]:
sample = cfg.samples[0]
sample_out = cfg.output_dir / sample.platform.lower()
sample_out.mkdir(parents=True, exist_ok=True)

adata = load_spatialdata_adata(
    sample.zarr_path,
    platform=sample.platform,
    table_key=sample.table_key,
    shape_key=sample.shape_key,
)
adata

Save the same QC outputs as the pipeline.

In [ ]:
qc_plot = plot_qc_histograms(
    adata,
    sample_out / f"{sample.sample_id}_qc_histograms.png",
    sample_label=sample.sample_id,
    platform=sample.platform,
    dpi=cfg.figure_dpi,
)
save_qc_metrics(adata, sample_out / f"{sample.sample_id}_qc_metrics.csv")
display(Image(filename=qc_plot))

In [ ]:
cfg

Run the exact Scanpy clustering function with the pipeline arguments. Change `cfg.min_counts`, `cfg.min_cells`, `cfg.n_neighbors`, or `cfg.leiden_resolution` above and rerun from here.

In [ ]:
clustered = run_scanpy_clustering(
    adata,
    min_counts=cfg.min_counts,
    min_cells=cfg.min_cells,
    normalize_target_sum=1, #cfg.normalize_target_sum,
    n_pcs=50,#cfg.n_pcs,
    n_neighbors=cfg.n_neighbors,
    leiden_resolution=0.3, #cfg.leiden_resolution,
    random_seed=cfg.random_seed,
)
clustered

In [ ]:
adata

Save the UMAP, spatial scatter, and clustered AnnData exactly as the pipeline does.

In [ ]:
umap_plot = plot_umap(
    clustered,
    sample_out / f"{sample.sample_id}_umap.png",
    dpi=cfg.figure_dpi,
)
spatial_plot = plot_spatial_scatter(
    clustered,
    sample_out / f"{sample.sample_id}_spatial_scatter_leiden.png",
    point_size=cfg.spatial_point_size,
    dpi=cfg.figure_dpi,
)
save_clustered_adata(
    clustered,
    sample_out / f"{sample.sample_id}_clustered.h5ad",
)
display(Image(filename=umap_plot))
display(Image(filename=spatial_plot))

Run the full pair stage function with the same config object.

In [ ]:
run_clustering_squidpy(cfg)